In [1]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import ftfy
import html
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
n_nan_source = df['source'].isna().sum()
n_empty_soruce = df['source'].astype(str).str.strip().eq('').sum()
n_placeholders_source = df['source'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_source}")
print(f"Number of empty rows: {n_empty_soruce}")
print(f"Number of placeholders (\\N): {n_placeholders_source}")
df['source'] = df['source'].replace('\\N', 'Unknown')
source_counts = df['source'].value_counts()
selected_sources = source_counts[source_counts >= 50].index.to_list()
selected_sources.remove('Unknown')
print(f"Relevant Sources:\n{selected_sources}")
coverage = source_counts[selected_sources].sum() / len(df)
print(f"Number of selected sources: {len(selected_sources)}")
print(f"Percentage of selected sources: {coverage:.2%}")

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 294
Relevant Sources:
['Yahoo', 'Reuters', 'BBC', 'New', 'Washington', 'RedNova', 'Boston', 'CNN', 'CNET', 'Topix.Net', 'Guardian', 'Motley', 'Register', 'International', 'Forbes', 'Time', 'ABC', 'InfoWorld', 'San', 'Wired', 'Xinhua', 'Computerworld', 'News', 'CSMonitor', 'PCWorld', 'Bloomberg', 'Seattle', 'Ananova', 'Syfy.com', 'Voice', 'USA', 'Independent', 'Scotsman', 'CBS', 'Rediff', 'Times', 'Channel', 'CBC', 'Newsday', 'Newsweek', 'Houston', 'Australian', 'Daily', 'Telegraph.co.uk', 'ESPN', 'Canada.com', 'BCC', 'Sports', 'Search', 'Chicago', 'Turkish', 'CNN/SI', 'MSNBC', 'London', 'National', 'Financial', 'Toronto', 'Indianapolis', 'Melbourne', 'Christian', 'Detroit', 'ZDNet.com', 'CTV', 'PC', 'ic', 'NEWS.com.au', 'RTE', 'Scotland', 'Hindustan', 'NPR', 'Al-Jazeera', 'Information', 'IPS', 'TechNewsWorld', 'News24', 'sportinglife.com', 'Arizona', 'Age', 'Taipei', 'Radio', 'Gulf', 'Sun-Sentinel.com', 'Indian'

### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(20))

Number of NaN rows: 1
Number of empty rows: 2
Number of placeholders (\N): 0
Titles Sample:
Id
79655                                           Surveying the Middle East Peace Process
21059                                                 John McEnroe's talk show canceled
23382                               Airstrikes kill dozen Taliban in Afghan south: U.S.
44315                                EU consultants list aims to separate good from bad
56870                                              Amazon Says Profit Jumped in Quarter
75916                                  Hungary PM Set for Victory Over Citizenship Vote
40523                                                          The year of the Gunners?
8962                            Italian railworkers to strike over network safety (AFP)
67938                                     Florida notebook: Focus on defense after loss
75105                                          Apple's iPhone to hit stores, lines grow
31722                    

### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of empty rows: 7
Number of placeholders (\N): 1874
Articles Sample
Id
43312                                                                            KABUL (Reuters) - A top rival to Afghan election  frontrunner President Hamid Karzai called off his boycott of  the process on Wednesday, making it likely that the historic  poll's result would be recognized by all despite voting  irregularities.   
8572                                                                                                                                                       A WONDER strike from Steven Gerrard looked like clinching all three points for Liverpool last night - until a last-gasp leveller from Lomano LuaLua ruined the Kop&#39;s party.
5322                                                                                                                                                                AP - Kristy Swanson pressed assault charges on Sunday against the ex-wife o

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
n_placeholders_pr = df['page_rank'].astype(str).str.strip().eq('\\N').sum()
rank_5=np.array([df['page_rank'].values==5]).sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")
print(df['page_rank'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number of articles with PageRank 5: 73891
Id
470      5
77047    5
14133    5
5246     5
12060    5
39068    5
77112    5
57902    5
70170    5
21113    5
Name: page_rank, dtype: int64


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
n_placeholders_time = df['timestamp'].astype(str).str.strip().eq('\\N').sum()
n_uslesess_time=np.array([df['timestamp'].values=="0000-00-00 00:00:00"]).sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
43554    0000-00-00 00:00:00
58694    2007-12-22 22:26:26
7322     0000-00-00 00:00:00
36663    2007-02-09 20:42:26
75189    0000-00-00 00:00:00
68815    2006-12-20 02:23:38
69018    0000-00-00 00:00:00
50865    2008-01-24 20:17:17
2619     2004-10-11 04:56:00
50121    2004-09-15 09:58:12
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [9]:
def process_timestamp(df):
    df = df.copy()
    
    df['dt_obj'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['has_date'] = df['dt_obj'].notna().astype(int)
    
    df['year'] = df['dt_obj'].dt.year.fillna(-1).astype(int)
    df['month'] = df['dt_obj'].dt.month.fillna(-1).astype(int)
    df['day_of_week'] = df['dt_obj'].dt.dayofweek.fillna(-1).astype(int)
    
    hours = df['dt_obj'].dt.hour
    df['time_of_day'] = pd.cut(hours, bins=[0, 6, 12, 18, 24], labels=[0, 1, 2, 3], right=False) 
    
    df['time_of_day'] = df['time_of_day'].astype('float').fillna(-1).astype(int)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    df = df.drop(columns=['timestamp', 'dt_obj'])
    
    return df

df = process_timestamp(df)

new_cols = ['has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend']
print(f"Nuove colonne aggiunte: {new_cols}")
print("Sample of 10 timestamps:")
print(df[new_cols].sample(10))

Nuove colonne aggiunte: ['has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend']
Sample of 10 timestamps:
       has_date  year  month  day_of_week  time_of_day  is_weekend
Id                                                                
48645         0    -1     -1           -1           -1           0
18736         1  2004     11            0            0           0
42455         1  2008      1            5            2           1
10078         0    -1     -1           -1           -1           0
29305         0    -1     -1           -1           -1           0
33515         0    -1     -1           -1           -1           0
6378          1  2007      2            2            3           0
39956         1  2007      7            3            0           0
55648         1  2008      1            3            0           0
45295         1  2008      1            4            2           0


### *Title* feature stemming

In [11]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_title(text):
    if pd.isna(text) or text == "": return ""
    text = str(text)
    
    text = html.unescape(text)
    text = ftfy.fix_text(text)
    text = text.lower()
    text = text.replace('.', '') 
    
    text = re.sub(r'(?:\\n|\s)*\(.*?\)\W*$', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s+(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    #  togliamo solo la punteggiatura pura.
    text = re.sub(r'[^a-z0-9]', ' ', text)

    words = text.split()
    
    meaningful_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) >= 2 and not w.isnumeric()]
    
    return " ".join(meaningful_words)

# Test rapido
def test_title(titles_series, cleaner_func, n):
    sample = titles_series.sample(n)
    print(f"Test on {n} random titles\n")
    for _, text in sample.items():
        cleaned = cleaner_func(text)
        print(f"Original text: {text}")
        print(f"Processed text: {cleaned}")
        print("-" * 50)

test_title(df['title'], clean_title, n=10)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/giorgiozoccatelli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Test on 10 random titles

Original text: Romanians vote in key contest
Processed text: romanian vote key contest
--------------------------------------------------
Original text: New DVDs
Processed text: new dvd
--------------------------------------------------
Original text: Jailed Palestinian Leader Drops Plans for Presidential Run
Processed text: jailed palestinian leader drop plan presidential run
--------------------------------------------------
Original text: Christmas Pilgrims in Bethlehem Voice Hope
Processed text: christmas pilgrim bethlehem voice hope
--------------------------------------------------
Original text: Gemina the 'crooked-necked giraffe' dies \
    (AP)\

Processed text: gemina crooked necked giraffe dy
--------------------------------------------------
Original text: 'Dying' Will Arrive Alive and Kicking
Processed text: dying arrive alive kicking
--------------------------------------------------
Original text: Bush: 'Great chance' to establish Palestinian st

### *Article* feature stemming

In [12]:
def clean_article(text):
    if pd.isna(text) or text == "" or str(text).strip() == "\\N": 
        return ""
    text = str(text)

    text = text[:1000] 

    text = html.unescape(text)
    text = re.sub(r'http[s]?://\S+', ' ', text)
    text = re.sub(r'\b[a-z0-9]+\.(com|net|org|gov)\b', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    trash_pattern = r'\b(src|href|alt|width|height|align|border|style|sig|valign|hspace|vspace)\b'
    text = re.sub(trash_pattern, ' ', text, flags=re.IGNORECASE)
    
    text = re.sub(r'\b[a-z]*\d{3,}[a-z]*\b', ' ', text)
    text = re.sub(r'\b[bcdfghjklmnpqrstvwxyz]{4,}\b', ' ', text) 

    text = ftfy.fix_text(text)
    text = text.lower()
    text = text.strip()
    
    text = re.sub(r'^\s*[a-z][\w\s,\.\(\)]{0,50}\s*--\s*', '', text)
    text = re.sub(r'^\s*[a-z][^\.\?!]{2,50}\s+[-–—]\s+', '', text)
    
    agencies_pattern = r'(?i)^\s*.*?\b(reuters|afp|ap|upi|bloomberg|bbc|cnn|blog)\b.*?\s*[-:–—]\s*'
    text = re.sub(agencies_pattern, '', text)

    text = re.sub(r'(?i)^by\s+[a-z\s\.,]+\s{2,}', '', text)
    text = text.replace('.', '')

    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s+(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z0-9]', ' ', text)
    words = text.split()
    
    meaningful_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) >= 2 and not w.isnumeric()]
    
    return " ".join(meaningful_words)

# Test
def test_article(article_series, cleaner_func, n):
    sample = article_series.sample(n)
    print(f"Test on {n} random articles\n")
    for _, text in sample.items():
        print(f"Original text (First 200 char): {str(text)}") 
        print(f"Processed text: {cleaner_func(text)}")
        print("-" * 50)

test_article(df['article'], clean_article, n=100)

Test on 100 random articles

Original text (First 200 char): BAGHDAD (Reuters) - Radical action is needed to save a "hollowed-out and fatally weakened" Iraqi state and ease violence that a new Pentagon report says is at an all-time high, a prominent think-tank warned on Tuesday.
Processed text: radical action needed save hollowed fatally weakened iraqi state ease violence new pentagon report say time high prominent think tank warned tuesday
--------------------------------------------------
Original text (First 200 char): In a race for the presidency, Hillary Rodham Clinton faces a problem that has dogged her since her days as first lady: an entrenched bloc of voters who simply do not like her.
Processed text: race presidency hillary rodham clinton face problem dogged since day first lady entrenched bloc voter simply like
--------------------------------------------------
Original text (First 200 char): <p><a href="http://us.rd.yahoo.com/dailynews/rss/entertainment/*http://news.yahoo.c

### *Title + Article* features merge

In [13]:
print(f"Starting shape: {df.shape}")
print(f"Starting columns: {df.columns.tolist()}")
df['title_clean'] = df['title'].apply(clean_title)
df['article_clean'] = df['article'].apply(clean_article)
df['text_combined'] = (df['title_clean'] + " " + df['title_clean'] + " " + df['article_clean']).str.strip()

n_empty = (df['text_combined'] == "").sum()
print(f"Removing {n_empty} rows with empty text")
df = df[df['text_combined'] != ""]
df = df.drop(columns=['title', 'article', 'title_clean', 'article_clean'])

print(f"Final shape: {df.shape}")
print(f"Actual columns: {df.columns.tolist()}")
print("Example of title + article combined:")
print(df['text_combined'].iloc[0])

Starting shape: (79997, 11)
Starting columns: ['source', 'title', 'article', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend']
Removing 3 rows with empty text
Final shape: (79994, 10)
Actual columns: ['source', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend', 'text_combined']
Example of title + article combined:
opec boost nigeria oil revenue 82m bpd opec boost nigeria oil revenue 82m bpd organisation petroleum exporting country opec hiking official output one million barrel per day effective november nigeria getting barrel per day per cent new quota


### Encoding categorical features

In [14]:
df['source'] = np.where(df['source'].isin(selected_sources), df['source'], 'Other')
categorical_cols = ['source']

print(f"Shape before encoding: {df.shape}")

df = pd.get_dummies(
    df, 
    columns=categorical_cols, 
    prefix=categorical_cols, 
    prefix_sep='_', 
    dtype=int
)

print(f"Shape after encoding: {df.shape}")
source_cols = [c for c in df.columns if c.startswith('source_')]
print(f"Number of generated features: {len(source_cols)}")

Shape before encoding: (79994, 10)
Shape after encoding: (79994, 109)
Number of generated features: 100


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from scipy.sparse import hstack 

target_col = 'label' 
custom_stop_words = [
    'said','say' ,'says','report', 'reported','according', 'today', 'yesterday', 'tomorrow', 'week', 'month', 'day','year', 'years', 'time']
my_stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words

y = df[target_col]
X = df.drop(columns=[target_col])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,
    stop_words=my_stop_words,
    sublinear_tf=True,
    norm='l2'
)

X_train_vectorized = tfidf.fit_transform(X_train['text_combined'])
X_test_vectorized = tfidf.transform(X_test['text_combined'])

X_train_base = X_train.drop(columns=['text_combined'])
X_test_base = X_test.drop(columns=['text_combined'])

X_train_final = hstack([X_train_vectorized, X_train_base])
X_test_final = hstack([X_test_vectorized, X_test_base])

print(f"Matrix Train: {X_train_final.shape}")
print(f"Matrix Test:  {X_test_final.shape}")

Train set: (63995, 108)
Test set:  (15999, 108)
Matrix Train: (63995, 30107)
Matrix Test:  (15999, 30107)


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score
import time

xgb_model = XGBClassifier(
    n_estimators=2000,          
    learning_rate=0.05,         
    max_depth=7,                
    colsample_bytree=0.4,       
    subsample=0.8,              
    min_child_weight=2,         
    reg_alpha=0.5,              
    reg_lambda=1.5,             
    objective='multi:softprob', 
    num_class=len(y.unique()),  
    n_jobs=-1,                  
    random_state=42,
    tree_method='hist',
    early_stopping_rounds=50          
)

print("Avvio training XGBoost (Configurazione Text-Focused)...")
start_time = time.time()

xgb_model.fit(
    X_train_final, y_train,
    eval_set=[(X_train_final, y_train), (X_test_final, y_test)],   
    verbose=100                 
)

end_time = time.time()
print(f"\nTraining completato in {(end_time - start_time)/60:.1f} minuti.")

print("\n--- RISULTATI XGBOOST ---")
y_pred_xgb = xgb_model.predict(X_test_final)

print(classification_report(y_test, y_pred_xgb))
print(f"MACRO F1 SCORE (XGBoost): {f1_score(y_test, y_pred_xgb, average='macro'):.4f}")

Avvio training XGBoost (Configurazione Text-Focused)...
[0]	validation_0-mlogloss:1.87871	validation_1-mlogloss:1.87923
